# ওষুধ রিমাইন্ডার: বাংলা ও হাতের লেখা OCR ট্রেনিং (Bangla + handwriting OCR training)

PaddleOCR-এর লেখা-পড়ার মডেলকে বাংলা ডোজ (১+০+১), "খাবারের আগে", "৭ দিন", ওষুধের নাম আর **হাতে লেখা**
প্রেসক্রিপশন পড়তে শেখানো হবে। পুরো মডেল GPU-তে ট্রেন হয়, তাই হাতের লেখায় সার্ভারের CPU-তে
ট্রেন করা মডেলের চেয়ে ভালো হওয়ার কথা।

**কী করবেন**
1. উপরে **Runtime → Change runtime type → T4 GPU** বেছে **Save**। (নিচে ডানে "T4" লেখা থাকতে হবে।)
2. **Runtime → Run all**।
3. প্রথম ঘরে Google Drive-এর অনুমতি চাইবে: **Connect / Allow** দিন। মডেলটি Drive-এ সেভ হবে, তাই
   Colab বন্ধ হয়ে গেলেও হারাবে না।
4. প্রায় ২ ঘণ্টা ট্যাবটি খোলা রাখুন। শেষে `oshudh_ocr_model.zip` ডাউনলোড হবে, আর Drive-এর
   **My Drive**-এও থাকবে। ফাইলটি Claude-কে পাঠান।

Steps: GPU runtime → Run all → allow Google Drive → keep the tab open (~2 h) → send
`oshudh_ocr_model.zip` (downloaded, and saved in My Drive). No real patient data is used: the
training images are generated.

## 1. GPU আর Google Drive (Check the GPU, connect Drive)

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "GPU নেই: Runtime → Change runtime type → T4 GPU → Save, তারপর আবার Run all"
print("GPU:", gpu)
# The trained model is copied to Drive at the end, so it survives the Colab session ending
from google.colab import drive
drive.mount("/content/drive")

## 2. দরকারি জিনিস ইনস্টল (Install tools)

In [ ]:
!apt-get -qq install -y libraqm0 libfribidi0 zip > /dev/null
!pip -q install --upgrade pillow fonttools
import PIL, importlib; importlib.reload(PIL)
from PIL import features
print("Bangla text shaping (raqm):", features.check("raqm"))

## 3. ফন্ট ডাউনলোড: বাংলা আর হাতের লেখা (Fonts: Bangla and handwriting)

In [ ]:
import os, urllib.request
BASE = "https://raw.githubusercontent.com/google/fonts/main"
def fetch(fonts, folder):
    os.makedirs(folder, exist_ok=True)
    for name, path in fonts.items():
        for licence in ("ofl", "apache"):
            try:
                urllib.request.urlretrieve(f"{BASE}/{licence}/{path}", f"{folder}/{name}")
                break
            except Exception:
                pass
        else:
            print("skipped", name)
    print(folder, len(os.listdir(folder)), "fonts")
fetch({
    "NotoSansBengali.ttf": "notosansbengali/NotoSansBengali%5Bwdth%2Cwght%5D.ttf",
    "NotoSerifBengali.ttf": "notoserifbengali/NotoSerifBengali%5Bwdth%2Cwght%5D.ttf",
    "HindSiliguri-Regular.ttf": "hindsiliguri/HindSiliguri-Regular.ttf",
    "HindSiliguri-Medium.ttf": "hindsiliguri/HindSiliguri-Medium.ttf",
    "HindSiliguri-Bold.ttf": "hindsiliguri/HindSiliguri-Bold.ttf",
    "AnekBangla.ttf": "anekbangla/AnekBangla%5Bwdth%2Cwght%5D.ttf",
    "BalooDa2.ttf": "balooda2/BalooDa2%5Bwght%5D.ttf",
    "Galada-Regular.ttf": "galada/Galada-Regular.ttf",
    "Mina-Regular.ttf": "mina/Mina-Regular.ttf",
    "TiroBangla-Regular.ttf": "tirobangla/TiroBangla-Regular.ttf",
    "Caveat.ttf": "caveat/Caveat%5Bwght%5D.ttf",
    "IndieFlower.ttf": "indieflower/IndieFlower-Regular.ttf",
    "ShadowsIntoLight.ttf": "shadowsintolight/ShadowsIntoLight.ttf",
    "PatrickHand.ttf": "patrickhand/PatrickHand-Regular.ttf",
    "GochiHand.ttf": "gochihand/GochiHand-Regular.ttf",
    "ArchitectsDaughter.ttf": "architectsdaughter/ArchitectsDaughter-Regular.ttf",
    "Handlee.ttf": "handlee/Handlee-Regular.ttf",
    "NanumPen.ttf": "nanumpenscript/NanumPenScript-Regular.ttf",
    "HomemadeApple.ttf": "homemadeapple/HomemadeApple-Regular.ttf",
    "JustAnotherHand.ttf": "justanotherhand/JustAnotherHand-Regular.ttf",
    "Kalam.ttf": "kalam/Kalam-Regular.ttf",
    "Sacramento.ttf": "sacramento/Sacramento-Regular.ttf",
}, "fonts")
fetch({
    "Atma-Regular.ttf": "atma/Atma-Regular.ttf",
    "ReenieBeanie.ttf": "reeniebeanie/ReenieBeanie.ttf",
}, "fonts_holdout")

## 4. প্রশিক্ষণের ছবি তৈরি (Generate training images)
প্রেসক্রিপশনের মতো লাইন: ওষুধের নাম, বাংলা ডোজ, সময়, দিন; ছাপা আর হাতের লেখার মতো, ফোনের ছবির মতো ঝাপসা/বাঁকা করে।

In [ ]:
%%writefile synth_rec.py
"""Synthetic line images for the PaddleOCR recognition model.

Real prescriptions write doses and instructions in Bangla script ("১+০+১  খাবারের আগে  ৭ দিন"),
which the pretrained models cannot read. Rendering such lines with Bangla fonts, then degrading them
like phone photos, gives the model thousands of correct examples before real data exists.

    uv run --extra ocr python -m oshudh_ml.synth_rec --fonts fonts/ --out data/synth --count 20000

Use several fonts, including handwriting-style ones (for Bangla, e.g. Noto Sans/Serif Bengali,
Hind Siliguri, Galada, Mina; for Latin, any handwriting font). Output is PaddleOCR's recognition
format: `images/*.jpg` and `train.txt` / `val.txt` lines of `images/name.jpg<TAB>text`.
"""

import argparse
import io
import random
import re
from collections.abc import Iterator
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter, ImageFont, features

BANGLA_DIGITS = "০১২৩৪৫৬৭৮৯"
FORMS = ["Tab.", "Tab", "Cap.", "Cap", "Syp.", "T.", "C."]
BRANDS = [
    "Napa", "Napa Extra", "Ace", "Seclo", "Sergel", "Maxpro", "Nexum", "Esonix", "Pantonix",
    "Fexo", "Alatrol", "Monas", "Zimax", "Azithrocin", "Cef-3", "Moxacil", "Ciprocin", "Filmet",
    "Amdocal", "Camlodin", "Osartil", "Angilock", "Comet", "Rosuva", "Atova", "Ecosprin",
    "Calbo D", "Rolac", "Sedil", "Disopan", "Orsaline-N", "Losectil", "Finix", "Tridosil",
    "Indever", "Bislol", "Concor", "Neuro-B", "Vonal", "Omidon", "Motigut", "Flugal",
]  # fmt: skip
STRENGTHS = [
    "5 mg",
    "10 mg",
    "20 mg",
    "40 mg",
    "50 mg",
    "120 mg",
    "500 mg",
    "0.5 mg",
    "500mg",
    "20mg",
]
PATTERNS = ["1+0+1", "1+1+1", "0+0+1", "1+0+0", "0+1+0", "1+1+1+1", "½+0+½", "1+0+½", "2+2+2"]
TIMINGS_BN = ["খাবারের আগে", "খাবারের পরে", "খালি পেটে", "ঘুমানোর আগে", "ভরা পেটে", "খাবারের সাথে"]
TIMINGS_LATIN = ["khabar age", "khabarer pore", "before meal", "after meal", "AC", "PC", "HS"]
DURATIONS_BN = ["দিন", "মাস", "সপ্তাহ"]
DURATIONS_LATIN = ["days", "din", "month", "weeks"]
EXTRAS_BN = ["চলবে", "প্রয়োজনে", "জ্বর হলে", "ব্যথা হলে", "জ্বর হলে খাবেন", "ব্যথা হলে খাবেন"]
FREQUENCIES = ["BD", "TDS", "QID", "OD", "HS", "SOS"]
# Common courses first, plus values that use every digit (০-৯): a digit never seen in training is
# a digit the model cannot read
DAYS = [3, 5, 7, 7, 10, 14, 15, 30, 30, 1, 2, 4, 6, 8, 9, 12, 20, 21, 28, 45, 60, 90]


def to_bangla_digits(text: str) -> str:
    return text.translate(str.maketrans("0123456789", BANGLA_DIGITS))


# PP-OCRv4's recognition config caps labels at 25 characters in three places (Global, the NRTR head
# and RecConAug); longer lines would be dropped silently, so the generator stays within it.
MAX_CHARS = 25


def sample_line(rng: random.Random) -> str:
    """One realistic prescription line of at most MAX_CHARS characters."""
    while True:
        line = _sample_line(rng)
        if len(line) <= MAX_CHARS:
            return line


def _sample_line(rng: random.Random) -> str:
    kind = rng.random()
    if kind < 0.3:  # medicine line
        # Handwritten lists often use a dot instead of a number, and put the strength in brackets
        number = rng.choice([f"{rng.randint(1, 6)}. ", f"{rng.randint(1, 6)}. ", ". ", ""])
        strength = rng.choice(STRENGTHS)
        strength = rng.choice(
            [f" {strength}", f" {strength}", f" ({strength.replace(' ', '')})", ""]
        )
        return f"{number}{rng.choice(FORMS)} {rng.choice(BRANDS)}{strength}"
    pattern = rng.choice(PATTERNS)
    days = str(rng.choice(DAYS))
    if kind < 0.85:  # Bangla instruction line (the main thing the pretrained model lacks)
        parts = [to_bangla_digits(pattern)]
        if rng.random() < 0.8:
            parts.append(rng.choice(TIMINGS_BN))
        if rng.random() < 0.7:
            parts.append(f"{to_bangla_digits(days)} {rng.choice(DURATIONS_BN)}")
        elif rng.random() < 0.3:
            parts.append(rng.choice(EXTRAS_BN))
        if rng.random() < 0.15:
            parts[0] += "-"  # "১+০+১-": a dash before the rest of the instruction
        return "  ".join(parts) + (";" if rng.random() < 0.1 else "")
    # Latin instruction line
    first = pattern if rng.random() < 0.7 else f"1 {rng.choice(FREQUENCIES)}"
    if rng.random() < 0.15:
        first += "-"
    parts = [first]
    if rng.random() < 0.7:
        parts.append(rng.choice(TIMINGS_LATIN))
    if rng.random() < 0.7:
        parts.append(f"{'x ' if rng.random() < 0.3 else ''}{days} {rng.choice(DURATIONS_LATIN)}")
    return "  ".join(parts)


def render(text: str, font: ImageFont.FreeTypeFont, rng: random.Random) -> Image.Image:
    """The line as a phone camera might capture it: tilt, blur, uneven light, noise, and the low
    resolution and heavy JPEG compression of a picture forwarded on a messenger and picked from the
    gallery; and, as handwriting often is, with wide gaps between words, slanted and with a thicker
    pen."""
    _, top, _, bottom = font.getbbox(text)
    pad = rng.randint(6, 16)
    # Words drawn one by one, so gaps can be wider than a space (the label keeps single spaces)
    spread = rng.uniform(1.5, 3.5) if rng.random() < 0.35 else 1.0
    pieces = [p for p in re.split(r"( +)", text) if p]
    widths = [font.getlength(p) * (spread if p.isspace() else 1) for p in pieces]
    w, h = int(sum(widths)) + 2 * pad, bottom - top + 2 * pad
    paper = rng.randint(215, 255)
    img = Image.new("L", (max(w, 8), max(h, 8)), paper)
    ink = rng.randint(0, 80)
    draw = ImageDraw.Draw(img)
    x = float(pad)
    for piece, width in zip(pieces, widths, strict=True):
        if not piece.isspace():
            draw.text((x, pad - top), piece, fill=ink, font=font)
        x += width
    if rng.random() < 0.3:  # slant
        shear = rng.uniform(-0.25, 0.25)
        extra = int(abs(shear) * img.height) + 1
        img = img.transform(
            (img.width + extra, img.height),
            Image.Transform.AFFINE,
            (1, shear, -extra / 2 if shear > 0 else 0, 0, 1, 0),
            fillcolor=paper,
        )
    if rng.random() < 0.2:  # thicker pen (ink is dark: spread the minimum)
        img = img.filter(ImageFilter.MinFilter(3))
    if rng.random() < 0.7:
        img = img.rotate(rng.uniform(-3, 3), expand=True, fillcolor=paper)
    blur = rng.random()
    if blur < 0.15:  # out of focus, still readable to a person
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(1.2, 2.0)))
    elif blur < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 1.2)))
    if rng.random() < 0.25:  # low resolution, scaled back up
        small = rng.uniform(0.45, 0.75)
        size = img.size
        img = img.resize((max(int(size[0] * small), 8), max(int(size[1] * small), 8)))
        img = img.resize(size, Image.Resampling.BICUBIC)
    if rng.random() < 0.5:  # noise
        noise = Image.effect_noise(img.size, rng.uniform(5, 25))
        img = Image.blend(img, noise, rng.uniform(0.05, 0.15))
    if rng.random() < 0.3:  # squash, as with an oblique photo
        img = img.resize((int(img.width * rng.uniform(0.8, 1.1)), img.height))
    if rng.random() < 0.3:  # recompressed JPEG
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=rng.randint(25, 60))
        img = Image.open(io.BytesIO(buf.getvalue()))
    return img.convert("RGB")


def coverage(path: Path) -> set[str]:
    """Characters the font can draw (a missing glyph would render as a box and teach wrong text)."""
    from fontTools.ttLib import TTFont

    with TTFont(str(path), lazy=True) as font:
        return {chr(cp) for cp in font.getBestCmap()}


def generate(
    fonts: list[Path], count: int, seed: int = 7, sizes: tuple[int, int] = (26, 44)
) -> Iterator[tuple[Image.Image, str]]:
    rng = random.Random(seed)
    loaded: dict[tuple[Path, int], ImageFont.FreeTypeFont] = {}
    covers = {path: coverage(path) for path in fonts}
    made = 0
    while made < count:
        text = sample_line(rng)
        needed = set(text) - {" "}
        usable = [p for p in fonts if needed <= covers[p]]
        if not usable:
            continue
        made += 1
        path, size = rng.choice(usable), rng.randint(*sizes)
        font = loaded.setdefault((path, size), ImageFont.truetype(str(path), size))
        yield render(text, font, rng), text


def write_dataset(fonts: list[Path], out: Path, count: int, seed: int = 7, val_share: float = 0.05):
    (out / "images").mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed + 1)
    with (
        (out / "train.txt").open("w", encoding="utf-8") as train,
        (out / "val.txt").open("w", encoding="utf-8") as val,
    ):
        for i, (img, text) in enumerate(generate(fonts, count, seed)):
            name = f"images/synth_{i:06d}.jpg"
            img.save(out / name, quality=rng.randint(60, 95))
            (val if rng.random() < val_share else train).write(f"{name}\t{text}\n")


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser()
    p.add_argument("--fonts", type=Path, required=True, help="Folder of .ttf/.otf fonts")
    p.add_argument("--out", type=Path, required=True)
    p.add_argument("--count", type=int, default=20000)
    p.add_argument("--seed", type=int, default=7)
    args = p.parse_args(argv)
    fonts = sorted([*args.fonts.glob("*.ttf"), *args.fonts.glob("*.otf")])
    if not fonts:
        p.error(f"no fonts in {args.fonts}")
    if not features.check("raqm"):
        # Without complex text layout, Bangla vowel signs and conjuncts are drawn in the wrong
        # order and the model would learn broken text
        p.error("Pillow has no libraqm: install it (apt-get install libraqm0) and reinstall Pillow")
    write_dataset(fonts, args.out, args.count, args.seed)
    print(f"{args.count} synthetic lines from {len(fonts)} fonts → {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
COUNT = 50000  # more is better; 50k takes a few minutes
!python synth_rec.py --fonts fonts --out data/synth --count $COUNT
!python synth_rec.py --fonts fonts_holdout --out data/holdout --count 600 --seed 99
!head -5 data/synth/train.txt

In [ ]:
# A few samples, to check that Bangla is drawn correctly (joined letters, not boxes)
from IPython.display import display
from PIL import Image
for line in open("data/synth/train.txt", encoding="utf-8").readlines()[:8]:
    path, text = line.rstrip("\n").split("\t", 1)
    print(text); display(Image.open(f"data/synth/{path}"))

## 5. মডেল ট্রেনিং (Train, about 1.5–2 hours on a T4)
ট্যাব খোলা রাখুন। EPOCHS কমালে তাড়াতাড়ি শেষ হবে, তবে কম শিখবে।

In [ ]:
%%writefile train_rec_colab.sh
#!/usr/bin/env bash
# Fine-tunes PaddleOCR's English PP-OCRv4 recognition model on prescription lines (Latin + Bangla),
# then exports it to ONNX for the backend. Run on a GPU machine, e.g. a free Google Colab GPU:
#
#   !git clone https://github.com/rahat1517/Osudh-Remainder && cd Osudh-Remainder/ml
#   !bash paddleocr/train_rec_colab.sh data/rec-v1 data/synth
#
# Inputs (PaddleOCR recognition format, made by oshudh_ml.export_rec and oshudh_ml.synth_rec):
#   <dir>/images/*.jpg, <dir>/train.txt, <dir>/val.txt   ("images/x.jpg<TAB>text" per line)
# Output: out/rec.onnx and out/dict.txt → copy to backend/models/ (see backend/models/README.md).
set -euo pipefail

DATA_DIRS=("$@")
[ ${#DATA_DIRS[@]} -ge 1 ] || { echo "usage: $0 <dataset dir>..." >&2; exit 2; }
EPOCHS=${EPOCHS:-30}
WORK=${WORK:-work}
OUT=${OUT:-out}
mkdir -p "$WORK" "$OUT"
WORK=$(realpath "$WORK"); OUT=$(realpath "$OUT")

# 1. A Python 3.11 environment of its own: PaddlePaddle 2.6 has no build for newer Pythons, and
#    Colab moves on (it ships Python 3.13 now). uv downloads that Python in seconds.
pip install -q uv
VENV="$WORK/py311"
[ -x "$VENV/bin/python" ] || uv venv -q --python 3.11 "$VENV"
PY="$VENV/bin/python"
PIP=(uv pip install -q --python "$PY")
# PaddlePaddle 2.6 is built against NumPy 1.x and dlopens cuDNN 8, which newer Colab images lack:
# install both into the environment and put cuDNN on the library path
# setuptools: PaddlePaddle imports it at start-up, and uv environments do not include it
"${PIP[@]}" "numpy<2" "paddlepaddle-gpu==2.6.2" paddle2onnx==1.3.1 "nvidia-cudnn-cu12==8.9.7.29" setuptools \
  onnx
CUDNN_LIB=$("$PY" -c "import nvidia.cudnn, os; print(os.path.join(nvidia.cudnn.__path__[0], 'lib'))")
export LD_LIBRARY_PATH="$CUDNN_LIB:${LD_LIBRARY_PATH:-}"
[ -d "$WORK/PaddleOCR" ] || git clone -q --depth 1 -b release/2.9 https://github.com/PaddlePaddle/PaddleOCR "$WORK/PaddleOCR"
"${PIP[@]}" -r "$WORK/PaddleOCR/requirements.txt" "numpy<2"
# Fail in the first minutes, with a clear message, if PaddlePaddle cannot use the GPU
"$PY" -c "import paddle; assert paddle.device.is_compiled_with_cuda(), 'PaddlePaddle without CUDA'; paddle.utils.run_check()"

# 2. Merge datasets (absolute image paths) and build the character dictionary: the pretrained
#    English characters first, in their order, then the new ones (Bangla...) found in the labels
TRAIN="$WORK/train.txt"; VAL="$WORK/val.txt"; : > "$TRAIN"; : > "$VAL"
for d in "${DATA_DIRS[@]}"; do
  abs=$(cd "$d" && pwd)
  sed "s#^#$abs/#" "$d/train.txt" >> "$TRAIN"
  [ -f "$d/val.txt" ] && sed "s#^#$abs/#" "$d/val.txt" >> "$VAL"
done
EN_DICT="$WORK/PaddleOCR/ppocr/utils/en_dict.txt"
"$PY" - "$TRAIN" "$VAL" "$EN_DICT" "$OUT/dict.txt" <<'PY'
import sys
train, val, en_dict, out = sys.argv[1:5]
chars = set()
for path in (train, val):
    for line in open(path, encoding="utf-8"):
        chars.update(line.rstrip("\n").split("\t", 1)[1])
chars.discard(" ")
# Every Bangla character, not just those in the labels: real prescriptions use words the
# synthetic lines do not, and a character missing from the list can never be read
import unicodedata
chars |= {chr(c) for c in range(0x0980, 0x0A00) if unicodedata.category(chr(c)) != "Cn"}
english = [c for c in open(en_dict, encoding="utf-8").read().splitlines() if c != " "]
new = sorted(chars - set(english))
open(out, "w", encoding="utf-8").write("\n".join(english + new) + "\n")
print(f"{len(english)} English + {len(new)} new characters: {''.join(new)}")
PY

# 3. Start from the pretrained English model and fine-tune
cd "$WORK"
[ -d en_PP-OCRv4_rec_train ] || {
  wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_train.tar
  tar xf en_PP-OCRv4_rec_train.tar
}
cd PaddleOCR
# The larger character list changes the shape of the output layers, and PaddleOCR would drop their
# pretrained weights and start them from random: the model would first forget all English. Copy
# each known character's weights to its new position instead; only new characters start fresh.
"$PY" - "$EN_DICT" "$OUT/dict.txt" "$WORK/en_PP-OCRv4_rec_train/best_accuracy" "$WORK/init" <<'PY'
import sys
import numpy as np
import paddle
sys.path.insert(0, ".")
from ppocr.data.imaug.label_ops import NRTRLabelEncode
from ppocr.postprocess.rec_postprocess import CTCLabelDecode

old_dict, new_dict, src, dst = sys.argv[1:5]
vocab = {
    "ctc": (CTCLabelDecode(old_dict, use_space_char=True).character,
            CTCLabelDecode(new_dict, use_space_char=True).character),
    "gtc": (NRTRLabelEncode(25, old_dict, use_space_char=True).character,
            NRTRLabelEncode(25, new_dict, use_space_char=True).character),
}
state = paddle.load(src + ".pdparams")
rng = np.random.default_rng(0)
for key, head, axis in [
    ("head.ctc_head.fc.weight", "ctc", 1),
    ("head.ctc_head.fc.bias", "ctc", 0),
    ("head.gtc_head.embedding.embedding.weight", "gtc", 0),
    ("head.gtc_head.tgt_word_prj.weight", "gtc", 1),
]:
    old, new = vocab[head]
    t = state[key].numpy()
    if t.shape[axis] == len(old) + 1:  # NRTR has one more token after the characters
        old, new = old + ["<extra>"], new + ["<extra>"]
    shape = list(t.shape)
    shape[axis] = len(new)
    # New characters start silent (tiny weights, the lowest bias): random ones fired everywhere
    # and scattered Bangla letters through English words until training suppressed them
    if t.ndim > 1:
        out = rng.normal(0, 0.01 * float(t.std()), shape).astype(t.dtype)
    else:
        out = np.full(shape, t.min(), t.dtype)
    index = {c: i for i, c in enumerate(old)}
    for j, c in enumerate(new):
        if c in index:
            if axis == 0:
                out[j] = t[index[c]]
            else:
                out[:, j] = t[:, index[c]]
    state[key] = paddle.to_tensor(out)
paddle.save(state, dst + ".pdparams")
print("pretrained weights kept for", sum(c in vocab["ctc"][0] for c in vocab["ctc"][1]), "characters")
PY
"$PY" tools/train.py -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec.yml -o \
  Global.pretrained_model="$WORK/init" \
  Global.character_dict_path="$OUT/dict.txt" \
  Global.use_space_char=true \
  Global.epoch_num="$EPOCHS" \
  Global.save_model_dir="$WORK/rec_output" \
  Train.dataset.data_dir=/ Train.dataset.label_file_list="[$TRAIN]" \
  Eval.dataset.data_dir=/ Eval.dataset.label_file_list="[$VAL]"

# 4. Export the best checkpoint and convert it to ONNX (what the backend's RapidOCR loads)
"$PY" tools/export_model.py -c configs/rec/PP-OCRv4/en_PP-OCRv4_rec.yml -o \
  Global.pretrained_model="$WORK/rec_output/best_accuracy" \
  Global.character_dict_path="$OUT/dict.txt" \
  Global.use_space_char=true \
  Global.save_inference_dir="$WORK/rec_inference"
"$VENV/bin/paddle2onnx" --model_dir "$WORK/rec_inference" --model_filename inference.pdmodel \
  --params_filename inference.pdiparams --save_file "$OUT/rec.onnx" --opset_version 11
# The pretrained weights contain many denormal (tiny) floats, which CPUs compute with very slowly:
# set them to zero (same results, about 2x faster on the server; see paddleocr/cpu/flush_denormals.py)
"$PY" - "$OUT/rec.onnx" <<'PY'
import sys
import numpy as np
import onnx
from onnx import numpy_helper
path = sys.argv[1]
model = onnx.load(path)
tensors = list(model.graph.initializer)
tensors += [a.t for n in model.graph.node if n.op_type == "Constant" for a in n.attribute if a.name == "value"]
flushed = 0
for t in tensors:
    w = numpy_helper.to_array(t)
    if w.dtype == np.float32:
        tiny = (w != 0) & (np.abs(w) < 1e-30)
        if tiny.any():
            flushed += int(tiny.sum())
            t.CopyFrom(numpy_helper.from_array(np.where(tiny, 0, w).astype(np.float32), t.name))
onnx.save(model, path)
print(f"{flushed} denormal weights set to zero")
PY
echo "Done: $OUT/rec.onnx and $OUT/dict.txt"

In [ ]:
!EPOCHS=15 bash train_rec_colab.sh data/synth 2>&1 | grep -vE "^\s*$" | tail -n 40

## 6. দ্রুত পরীক্ষা (Quick check on unseen lines and unseen fonts)

In [ ]:
!pip -q install rapidocr-onnxruntime
from rapidocr_onnxruntime import RapidOCR
engine = RapidOCR(rec_model_path="out/rec.onnx", rec_keys_path="out/dict.txt")
def check(folder, n=300):
    lines = []
    for name in ("val.txt", "train.txt"):
        if os.path.exists(f"{folder}/{name}"):
            lines += open(f"{folder}/{name}", encoding="utf-8").read().splitlines()
    lines = lines[:n]
    right = 0
    for line in lines:
        path, truth = line.split("\t", 1)
        result, _ = engine(f"{folder}/{path}", use_det=False, use_cls=False, use_rec=True)
        text = result[0][0] if result else ""
        right += text.replace(" ", "") == truth.replace(" ", "")
    print(f"{folder}: exact lines {right}/{len(lines)} = {right/len(lines):.0%}")
check("data/synth")
check("data/holdout")  # fonts never trained on, incl. a handwriting one

## 7. সেভ আর ডাউনলোড (Save to Drive, download, send to Claude)

In [ ]:
import shutil
!cd out && zip -q ../oshudh_ocr_model.zip rec.onnx dict.txt && ls -la ../oshudh_ocr_model.zip
shutil.copy("oshudh_ocr_model.zip", "/content/drive/MyDrive/oshudh_ocr_model.zip")
print("Saved in Google Drive: My Drive / oshudh_ocr_model.zip")
from google.colab import files
files.download("oshudh_ocr_model.zip")